# 01_extract_EU · da API do EMSC para a landing

**Lakehouse Sísmico Europeu · aula 2 de 4**

| | |
|---|---|
| **Entra** | API FDSN do EMSC / SeismicPortal (`/count` e `/query`) |
| **Sai** | 12 arquivos em `/Volumes/sismos_eu/bronze/landing/terremotos/escopo=XXXXXXXX/ano=2025/mes=MM.json` |
| **Também grava** | uma linha por arquivo em `sismos_eu.bronze.extracoes_manifesto`: o que a API disse, o que veio, qual filtro |
| **Regra** | rodar de novo sobrescreve os mesmos 12 arquivos, nunca duplica |

> Dados: EMSC-CSEM SeismicPortal, https://www.seismicportal.eu. Os termos do portal pedem crédito à fonte ao usar os dados.

As quatro decisões que este notebook carrega:
1. **Landing antes da tabela.** A resposta vai para o volume byte a byte. Se o EMSC revisar o passado amanhã, você ainda tem o que ele disse hoje.
2. **Fatia por mês, conferida.** Antes de cada mês, o `/count` diz quantos eventos virão, e o código confere se vieram todos. Diferente do USGS, a API do EMSC **não documenta um teto de eventos por consulta**: a conferência é o que protege contra um mês cortado. A janela de cada mês também é conferida: nenhum evento pode cair fora do mês, e nenhum `unid` pode se repetir.
3. **Caminho determinístico.** O nome do arquivo depende só da janela pedida **e do escopo do filtro** (magnitude mínima e região). Se alguém mudar o filtro, o caminho muda junto, e o Auto Loader enxerga arquivos novos em vez de ignorar conteúdo diferente com o mesmo nome.
4. **Falha que fica vermelha.** Retry para 429, 5xx e falha de rede (timeout, conexão caída). Um 4xx (incluindo o 413, "dados demais") é pedido errado, e repetir não conserta.

### O que muda em relação ao USGS
| | USGS | EMSC (este projeto) |
|---|---|---|
| Formato | `format=geojson` | `format=json` (o padrão da API é `xml`) |
| `/count` | `{count, maxAllowed}` | só `{"count": N}` |
| Cobertura | global | **global**, com melhor densidade na Europa: por isso há um **filtro geográfico** abaixo |
| Extensão do arquivo | `.geojson` | `.json` |

## 0 · Garantir a estrutura
Este notebook escreve no volume `landing` e numa tabela de manifesto. Ele garante tudo sozinho, sem depender de alguém ter
criado antes. Com `IF NOT EXISTS`, rodar de novo não muda nada.

O **manifesto** é o diário da extração: uma linha por arquivo gravado, com o `/count` que a API deu, quantos eventos vieram,
o tamanho em bytes e o filtro usado. O `02_bronze_EU` usa esse diário para conferir a Bronze contra o que a API disse na época.
Ele fica **fora** da landing de propósito: o Auto Loader leria qualquer arquivo que estivesse lá.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS sismos_eu;
CREATE SCHEMA  IF NOT EXISTS sismos_eu.bronze;
CREATE VOLUME  IF NOT EXISTS sismos_eu.bronze.landing;
CREATE TABLE   IF NOT EXISTS sismos_eu.bronze.extracoes_manifesto (
  escopo     STRING,
  ano        INT,
  mes        INT,
  esperado   BIGINT,
  recebidos  BIGINT,
  bytes      BIGINT,
  arquivo    STRING,
  filtro     STRING,
  extraido_em TIMESTAMP
);

## 1 · Parâmetros
O widget `ano` aparece no topo do notebook. Na aula 4, ele vira parâmetro do Job.

**`REGIAO` é uma decisão do projeto, não da API.** O catálogo do EMSC é global (o primeiro evento de julho de 2025 é
em Sumatra). O retângulo abaixo vai de 27°N a 72°N e de 32°W a 45°E. Ele inclui Açores, Islândia e Turquia, mas **também
partes do Oriente Médio e do norte da África**, e deixa de fora Svalbard (acima de 72°N). É um retângulo, não um mapa da
Europa. Ajuste os quatro números ou use `REGIAO = {}` para o mundo todo. A decisão final de "o que é Europa" pode ficar
para a Silver, por `flynn_region`.

**`ESCOPO`** reúne tudo que muda *quais* eventos vêm (magnitude mínima e região). Dele sai um código curto, o `ESCOPO_ID`,
que entra no caminho do arquivo. Mudou o filtro, mudou o código, mudou a pasta.

In [0]:
import hashlib
import json
import os
import time
from datetime import date, timedelta

import requests
from pyspark.sql import functions as F

dbutils.widgets.text("ano", "2025")
ANO = int(dbutils.widgets.get("ano"))

API     = "https://www.seismicportal.eu/fdsnws/event/1"
LANDING = "/Volumes/sismos_eu/bronze/landing/terremotos"
MANIFESTO = "sismos_eu.bronze.extracoes_manifesto"

REGIAO = {"minlatitude": 27, "maxlatitude": 72, "minlongitude": -32, "maxlongitude": 45}
ESCOPO = {"minmagnitude": 2.5, **REGIAO}
FILTRO = {"format": "json", "orderby": "time-asc", **ESCOPO}

ESCOPO_ID = hashlib.md5(json.dumps(ESCOPO, sort_keys=True).encode()).hexdigest()[:8]

print(f"ano={ANO}  escopo={ESCOPO_ID}")
print(f"-> {LANDING}/escopo={ESCOPO_ID}/ano={ANO}/")

ano=2025  escopo=9c121421
-> /Volumes/sismos_eu/bronze/landing/terremotos/escopo=9c121421/ano=2025/


## 2 · Conhecer a API antes de escrever o loop
Duas perguntas, uma chamada cada:
- **Quanto vem?** O `/count` responde sem baixar nenhum evento. Aqui ele só devolve `{"count": N}`.
- **Como vem?** Um `/query` com `limit=1` mostra a anatomia de um evento.

Primeiro, o efeito do filtro geográfico em julho: o mesmo mês, com e sem `REGIAO`.

In [0]:
julho_regiao = {**FILTRO, "starttime": f"{ANO}-07-01T00:00:00", "endtime": f"{ANO}-07-31T23:59:59.99999"}
julho_mundo  = {k: v for k, v in julho_regiao.items() if k not in REGIAO}

r_reg   = requests.get(f"{API}/count", params=julho_regiao, timeout=30)
r_mundo = requests.get(f"{API}/count", params=julho_mundo,  timeout=30)
print("região :", r_reg.status_code,   r_reg.text)
print("mundo  :", r_mundo.status_code, r_mundo.text)

região : 200 {"count":623}
mundo  : 200 {"count":7007}


Agora um evento só, de julho, para ver o formato:

In [0]:
amostra = requests.get(f"{API}/query", params={**julho_regiao, "limit": 1}, timeout=30).json()

print("chaves do topo:", list(amostra.keys()))
print("metadata:", amostra["metadata"])
print()
print(json.dumps(amostra["features"][0], indent=2, ensure_ascii=False))

chaves do topo: ['type', 'metadata', 'features']
metadata: {'count': 1}

{
  "type": "Feature",
  "geometry": {
    "type": "Point",
    "coordinates": [
      27.7089,
      39.2931,
      -10.1
    ]
  },
  "id": "20250701_0000002",
  "properties": {
    "source_id": "1826895",
    "source_catalog": "EMSC-RTS",
    "lastupdate": "2025-07-01T06:33:53.698741Z",
    "time": "2025-07-01T00:31:40.96Z",
    "flynn_region": "WESTERN TURKEY",
    "lat": 39.2931,
    "lon": 27.7089,
    "depth": 10.1,
    "evtype": "ke",
    "auth": "EMSC",
    "mag": 2.8,
    "magtype": "ml",
    "unid": "20250701_0000002"
  }
}


Repare, porque é assunto da aula 3:
- `metadata.count` conta os eventos **devolvidos** nesta resposta (aqui, 1 por causa do `limit`). Não é o total do período: para isso existe o `/count`.
- `time` e `lastupdate` são **texto ISO 8601** (`2025-07-01T00:31:40.96Z`), e não milissegundos como no USGS.
- `geometry.coordinates` é `[longitude, latitude, -profundidade]`: o terceiro valor vem **negativo**. Já `properties.depth` vem positiva, e `lat` e `lon` também aparecem em `properties`.
- `properties.evtype` é um **código** do tipo de evento (`ke` no exemplo), e não a palavra `earthquake`.
- `properties.auth` é o instituto que contribuiu com o evento. Cada instituto tem seu jeito de medir a magnitude (`magtype`).
- `unid` é o identificador do evento e é igual ao `id`; `source_id` é o número interno do EMSC.
- Não existe `status` (automático ou revisado). A revisão aparece como mudança em `lastupdate`.

## 3 · Uma chamada que sabe falhar
| Resposta | O que significa | O que fazer |
|---|---|---|
| `200` | deu certo | seguir |
| `429` | chamadas demais | esperar e tentar de novo |
| `5xx` | servidor com problema | esperar e tentar de novo |
| timeout, conexão caída | a resposta nem chegou | esperar e tentar de novo |
| `413` e outros `4xx` | o **pedido** está errado (o 413 é "dados demais") | falhar na hora: repetir não conserta |

A espera dobra a cada tentativa (*backoff exponencial*): 1, 2, 4, 8, 16 segundos.

In [0]:
def chamar(endpoint, params, tentativas=5):
    r, erro = None, None
    for i in range(tentativas):
        try:
            r = requests.get(f"{API}/{endpoint}", params=params, timeout=120)
        except requests.exceptions.RequestException as e:   # timeout, conexão caída...
            r, erro = None, e
            print(f"  {type(e).__name__} em /{endpoint}, nova tentativa em {2 ** i}s")
            time.sleep(2 ** i)
            continue
        if r.status_code == 429 or r.status_code >= 500:
            print(f"  {r.status_code} em /{endpoint}, nova tentativa em {2 ** i}s")
            time.sleep(2 ** i)
            continue
        r.raise_for_status()   # 413 e outros 4xx: falha na hora
        return r
    if r is None:
        raise erro             # esgotou as tentativas sem resposta nenhuma
    r.raise_for_status()       # esgotou as tentativas: falha com o último erro HTTP

## 4 · Um mês: perguntar, baixar, conferir, gravar
A função faz uma fatia inteira. Se qualquer conferência falhar, ela levanta erro **antes** de gravar:
arquivo pela metade nunca chega à landing. No fim, devolve uma linha para o manifesto.

Quatro cuidados que o EMSC pede:
- **A janela tem que ter hora explícita.** O `endtime` do EMSC é inclusivo e, quando recebe só a data (`2025-02-01`), tudo indica que cobre o **dia inteiro**.
  Na primeira versão deste notebook, "de 01-01 a 02-01" trazia o dia 1º de fevereiro dentro do arquivo de janeiro e de novo no de fevereiro: 461 eventos repetidos em 2025.
  Agora cada mês vai de `00:00:00` do dia 1º até `23:59:59.99999` do último dia.
- **A função confere a janela.** Depois do download, ela verifica que nenhum evento está fora do mês e que nenhum `unid` se repete no arquivo. Se algo falhar, levanta erro **antes** de gravar.
- Sem resultado, a API responde `204` sem corpo. Por isso o `/count` vem primeiro: se for zero, a função nem chama o `/query` e devolve `None`.
- O `/count` e o `/query` são duas chamadas. Para um mês que já passou, os dois concordam. No mês corrente, um evento novo pode chegar entre as duas, e a conferência acusa a diferença.

In [0]:
def limites_do_mes(ano, mes):
    """Primeiro e último instante do mês, com hora explícita (o endtime do EMSC é inclusivo)."""
    primeiro_do_proximo = date(ano + 1, 1, 1) if mes == 12 else date(ano, mes + 1, 1)
    ultimo_dia = primeiro_do_proximo - timedelta(days=1)
    return f"{ano}-{mes:02d}-01T00:00:00", f"{ultimo_dia.isoformat()}T23:59:59.99999"


def janela_do_mes(ano, mes):
    inicio, fim = limites_do_mes(ano, mes)
    return {**FILTRO, "starttime": inicio, "endtime": fim}


def extrair_mes(ano, mes):
    janela = janela_do_mes(ano, mes)

    esperado = chamar("count", janela).json()["count"]
    if esperado == 0:
        print(f"  {ano}-{mes:02d}: /count = 0, nada a gravar")
        return None

    resposta  = chamar("query", janela)
    eventos   = resposta.json()["features"]
    recebidos = len(eventos)
    if recebidos != esperado:
        raise ValueError(f"/count disse {esperado}, vieram {recebidos}")

    # a janela não pode vazar para o mês vizinho (comparação por texto ISO, até os segundos)
    inicio, fim = limites_do_mes(ano, mes)
    fora = [e["properties"]["time"] for e in eventos
            if not (inicio[:19] <= e["properties"]["time"][:19] <= fim[:19])]
    if fora:
        raise ValueError(f"{len(fora)} evento(s) fora da janela {inicio} a {fim}, por exemplo {fora[0]}")

    ids = [e["properties"]["unid"] for e in eventos]
    if len(set(ids)) != len(ids):
        raise ValueError(f"{len(ids) - len(set(ids))} unid repetido(s) no mesmo arquivo")

    arquivo = f"escopo={ESCOPO_ID}/ano={ano}/mes={mes:02d}.json"
    destino = f"{LANDING}/{arquivo}"
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    with open(destino, "wb") as f:
        f.write(resposta.content)   # byte a byte, como veio

    return {"escopo": ESCOPO_ID, "ano": ano, "mes": mes,
            "esperado": esperado, "recebidos": recebidos,
            "bytes": len(resposta.content), "arquivo": arquivo,
            "filtro": json.dumps(ESCOPO, sort_keys=True)}


COLUNAS_MANIFESTO = ["escopo", "ano", "mes", "esperado", "recebidos", "bytes", "arquivo", "filtro"]
SCHEMA_MANIFESTO  = ("escopo string, ano int, mes int, esperado bigint, recebidos bigint, "
                     "bytes bigint, arquivo string, filtro string")


def registrar(linhas):
    """Anexa ao manifesto as extrações que deram certo. É um diário: cada execução acrescenta linhas."""
    if not linhas:
        return
    dados = [tuple(l[c] for c in COLUNAS_MANIFESTO) for l in linhas]
    (spark.createDataFrame(dados, SCHEMA_MANIFESTO)
          .withColumn("extraido_em", F.current_timestamp())
          .write.mode("append").saveAsTable(MANIFESTO))


# teste com um mês só antes do loop (grava o arquivo, mas ainda não o manifesto)
print(extrair_mes(ANO, 7))

{'escopo': '9c121421', 'ano': 2025, 'mes': 7, 'esperado': 623, 'recebidos': 623, 'bytes': 336470, 'arquivo': 'escopo=9c121421/ano=2025/mes=07.json', 'filtro': '{"maxlatitude": 72, "maxlongitude": 45, "minlatitude": 27, "minlongitude": -32, "minmagnitude": 2.5}'}


## 5 · O ano inteiro
Um mês que falha **não para** os outros: o erro é anotado e o loop segue. Os meses que deram certo vão para o manifesto
**antes** do fim. Se a lista de falhas não estiver vazia, o notebook termina em vermelho. É isso que vai fazer o Job da aula 4 avisar alguém.

A pausa de 1 segundo entre meses é educação: a API é pública e não documenta limite de chamadas.

In [0]:
falhas, linhas = [], []
print("          /count  recebidos")
for mes in range(1, 13):
    try:
        linha = extrair_mes(ANO, mes)
        if linha is None:
            print(f"· {ANO}-{mes:02d}  vazio")
        else:
            linhas.append(linha)
            print(f"✓ {ANO}-{mes:02d}  {linha['esperado']:>6,}  {linha['recebidos']:>6,}".replace(",", "."))
    except Exception as erro:
        falhas.append((f"{ANO}-{mes:02d}", str(erro)))
        print(f"✗ {ANO}-{mes:02d}  {erro}")
    time.sleep(1)

registrar(linhas)

total = sum(l["recebidos"] for l in linhas)
print(f"total  {total:,} eventos em {len(linhas)} arquivos".replace(",", "."))
if falhas:
    raise Exception(f"{len(falhas)} mês(es) falharam: {falhas}")

          /count  recebidos
✓ 2025-01     757     757
✓ 2025-02   3.064   3.064
✓ 2025-03     789     789
✓ 2025-04     974     974
✓ 2025-05     768     768
✓ 2025-06     672     672
✓ 2025-07     623     623
✓ 2025-08   1.462   1.462
✓ 2025-09     846     846
✓ 2025-10   1.067   1.067
✓ 2025-11   1.022   1.022
✓ 2025-12     716     716
total  12.760 eventos em 12 arquivos


## 6 · Conferir a landing e o manifesto
Esperado: **12 arquivos** `.json`, um por mês, e 12 linhas novas no manifesto a cada execução completa.
Rode o notebook inteiro de novo: continuam 12 arquivos, e o manifesto ganha mais 12 linhas (é um diário).

In [0]:
arquivos = dbutils.fs.ls(f"{LANDING}/escopo={ESCOPO_ID}/ano={ANO}/")
for a in arquivos:
    print(f"{a.name:<16} {a.size / 1e6:>5.1f} MB")
print(f"{len(arquivos)} arquivos · {sum(a.size for a in arquivos) / 1e6:.1f} MB")

mes=01.json        0.4 MB
mes=02.json        1.7 MB
mes=03.json        0.4 MB
mes=04.json        0.5 MB
mes=05.json        0.4 MB
mes=06.json        0.4 MB
mes=07.json        0.3 MB
mes=08.json        0.8 MB
mes=09.json        0.5 MB
mes=10.json        0.6 MB
mes=11.json        0.6 MB
mes=12.json        0.4 MB
12 arquivos · 6.9 MB


In [0]:
display(spark.table(MANIFESTO)
        .where(F.col("escopo") == ESCOPO_ID)
        .orderBy(F.desc("extraido_em"), "mes")
        .limit(24))

escopo,ano,mes,esperado,recebidos,bytes,arquivo,filtro,extraido_em
9c121421,2025,1,757,757,409232,escopo=9c121421/ano=2025/mes=01.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,2,3064,3064,1684016,escopo=9c121421/ano=2025/mes=02.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,3,789,789,428340,escopo=9c121421/ano=2025/mes=03.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,4,974,974,526063,escopo=9c121421/ano=2025/mes=04.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,5,768,768,415418,escopo=9c121421/ano=2025/mes=05.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,6,672,672,363509,escopo=9c121421/ano=2025/mes=06.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,7,623,623,336470,escopo=9c121421/ano=2025/mes=07.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,8,1462,1462,788991,escopo=9c121421/ano=2025/mes=08.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,9,846,846,456652,escopo=9c121421/ano=2025/mes=09.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z
9c121421,2025,10,1067,1067,576074,escopo=9c121421/ano=2025/mes=10.json,"{""maxlatitude"": 72, ""maxlongitude"": 45, ""minlatitude"": 27, ""minlongitude"": -32, ""minmagnitude"": 2.5}",2026-11-22T22:16:46.745Z


### Para pensar antes do `02_bronze_EU`
Rodar este notebook de novo **sobrescreve** os 12 arquivos, possivelmente com dados revisados pelo EMSC
(o campo `lastupdate` muda quando um evento é revisado).
O Auto Loader vai perceber? A resposta aparece no fim do próximo notebook.